# Watershed iSnobal vs Noah-MP Replot Template

This notebook replots the **watershed-average SWE** and **watershed-average snow depth**
3×3 water-year figures using the current East River plotting template.

Template choices applied here:

- **No main figure title**
- **Two separate 3×3 figures**
- Shared figure-level x/y labels
- Legend inside the first subplot, upper left, with a visible box
- Panel spacing matched to the current preferred 3×3 template
- Water-year panel titles at 20 pt
- Statistics shown in the upper-right of each panel


In [ ]:
from __future__ import annotations

from pathlib import Path
import math

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

from east_river_workflow.config import load_config
from east_river_workflow.constants import MODEL_COLORS
from east_river_workflow.metrics import pearson_mean_bias
from east_river_workflow.plotting import configure_matplotlib
from east_river_workflow.utils import water_year_bounds

# -------------------------------------------------------------------------
# USER SETTING
# -------------------------------------------------------------------------
CONFIG_PATH = Path(
    "/bsuscratch/zihaozhang/iSnobal/East_River_Workflow_Code_v0.8.2/"
    "config/east_river_config.yaml"
)

## Variable definitions

This notebook reads the processed watershed daily tables:

- `tables/isnobal_watershed_daily.csv`
- `tables/noahmp_watershed_daily.csv`

and makes one 3×3 figure for each variable.


In [ ]:
VARIABLE_SPECS = {
    "SWE": {
        "isnobal_col": "swe_mm",
        "noah_col": "swe_mm",
        "ylabel": "SWE (mm)",
        "slug": "swe",
        "y_step": 50.0,
    },
    "Snow depth": {
        "isnobal_col": "snow_depth_mm",
        "noah_col": "snow_depth_mm",
        "ylabel": "Snow depth (mm)",
        "slug": "snow_depth",
        "y_step": 100.0,
    },
}

## Helper functions

In [ ]:
def paired_stats_by_date(
    reference: pd.DataFrame,
    model: pd.DataFrame,
    reference_col: str,
    model_col: str,
) -> dict[str, float | int]:
    """Pearson r and mean bias using finite date-matched pairs.

    Bias convention follows the East River workflow helper:
        model - reference

    For these watershed figures, reference = iSnobal and model = Noah-MP,
    so the reported mean bias is Noah-MP - iSnobal.
    """
    pair = (
        reference[["date", reference_col]]
        .rename(columns={reference_col: "reference"})
        .merge(
            model[["date", model_col]].rename(columns={model_col: "model"}),
            on="date",
            how="inner",
        )
    )

    return pearson_mean_bias(
        pair["reference"].to_numpy(),
        pair["model"].to_numpy(),
    )


def format_stats(stats: dict[str, float | int]) -> str:
    r = float(stats["pearson_r"])
    bias = float(stats["mean_bias"])

    rtxt = "NA" if not np.isfinite(r) else f"{r:.2f}"
    btxt = "NA" if not np.isfinite(bias) else f"{bias:+.1f} mm"

    return f"r = {rtxt}\nMB = {btxt}"


def nice_ymax(values: list[float], step: float) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]

    if not len(arr):
        return step

    top = float(np.nanmax(arr)) * 1.05
    return float(np.ceil(top / step) * step)


def format_water_year_axis(ax: plt.Axes, wy: int) -> None:
    """Format an Oct-Sep water-year x-axis."""
    import matplotlib.dates as mdates

    start, end = water_year_bounds(int(wy))
    ax.set_xlim(start, end)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.spines[["top", "right"]].set_visible(False)

## Plotting function

This function applies the current template:

- `figsize=(21, 5.5 * nrows)`
- `top=0.95`, `bottom=0.15`, `left=0.07`, `hspace=0.30`, `wspace=0.16`
- shared x label at `y=0.095`
- shared y label at `x=0.025`
- boxed legend inside the first subplot at `loc="upper left"`
- **no main title**


### Publication-quality plotting settings
This version uses 2.2-pt model lines, stronger axis/tick strokes, a boxed upper-left legend, 600-dpi raster export, and a companion vector PDF.


In [ ]:
def plot_watershed_variable(
    cfg,
    isnobal: pd.DataFrame,
    noah: pd.DataFrame,
    variable: str,
    show_figure: bool = True,
) -> tuple[Path, pd.DataFrame]:
    """Plot all configured water years for one watershed variable."""

    spec = VARIABLE_SPECS[variable]
    years = [int(y) for y in cfg.section("project")["water_years"]]

    i = isnobal.copy()
    n = noah.copy()

    ncols = 3
    nrows = math.ceil(len(years) / ncols)

    ymax = nice_ymax(
        [
            i[spec["isnobal_col"]].max(skipna=True),
            n[spec["noah_col"]].max(skipna=True),
        ],
        step=float(spec["y_step"]),
    )

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(21, 5.5 * nrows),
        sharey=True,
        squeeze=False,
    )

    stat_rows = []

    for ax, wy in zip(axes.flat, years):
        ii = i[i["water_year"].eq(wy)].sort_values("date")
        nn = n[n["water_year"].eq(wy)].sort_values("date")

        ax.plot(
            ii["date"],
            ii[spec["isnobal_col"]],
            color=MODEL_COLORS["iSnobal"],
            lw=2.5,
        )

        ax.plot(
            nn["date"],
            nn[spec["noah_col"]],
            color=MODEL_COLORS["Noah-MP"],
            lw=2.5,
        )

        stats = paired_stats_by_date(
            ii,
            nn,
            spec["isnobal_col"],
            spec["noah_col"],
        )

        ax.set_title(
            f"WY {wy}",
            fontsize=20,
            fontweight="normal",
            loc="center",
            pad=8,
        )

        ax.text(
            0.97,
            0.95,
            format_stats(stats),
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=max(18, cfg.section("plotting")["font_size"] - 4),
        )

        ax.set_ylim(0, ymax)
        format_water_year_axis(ax, wy)

        # Publication-quality axis styling.
        ax.spines["left"].set_linewidth(1.0)
        ax.spines["bottom"].set_linewidth(1.0)
        ax.tick_params(axis="both", which="major", width=1.0, length=5)

        stat_rows.append(
            {
                "water_year": wy,
                "variable": variable,
                "reference": "iSnobal",
                "model": "Noah-MP",
                "n": int(stats["n"]),
                "pearson_r": stats["pearson_r"],
                "mean_bias_mm": stats["mean_bias"],
            }
        )

    # Hide unused panels.
    for ax in axes.flat[len(years):]:
        ax.set_visible(False)

    # Shared figure-level labels.
    fig.supxlabel("Month", y=0.095)
    fig.supylabel(spec["ylabel"], x=0.015)

    # No main title by design.

    # Legend inside the first subplot, upper left, with a visible box.
    axes.flat[0].legend(
        handles=[
            Line2D(
                [], [], color=MODEL_COLORS["iSnobal"],
                lw=2.5, label="iSnobal"
            ),
            Line2D(
                [], [], color=MODEL_COLORS["Noah-MP"],
                lw=2.5, label="Noah-MP"
            ),
        ],
        loc="upper left",
        ncol=1,
        frameon=True,
        facecolor="white",
        edgecolor="0.4",
        framealpha=0.9,
        fancybox=False,
        borderpad=0.5,
        handlelength=2.2,
    )

    fig.subplots_adjust(
        top=0.95,
        bottom=0.15,
        left=0.07,
        hspace=0.30,
        wspace=0.16,
    )

    out_dir = cfg.output_dir / "figures" / "baseline"
    out_dir.mkdir(parents=True, exist_ok=True)

    ext = cfg.section("plotting")["figure_format"]
    out_path = (
        out_dir
        / f"watershed_mean_{spec['slug']}_replot.{ext}"
    )

    # Publication-quality raster export (600 dpi).
    fig.savefig(
        out_path,
        dpi=600,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    # Also save a vector PDF for journal submission when accepted.
    pdf_path = out_path.with_suffix(".pdf")
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    if show_figure:
        plt.show()
    else:
        plt.close(fig)

    return out_path, pd.DataFrame(stat_rows)

## Load the processed watershed daily tables

In [ ]:
cfg = load_config(CONFIG_PATH)
configure_matplotlib(cfg)

table_dir = cfg.output_dir / "tables"

isnobal = pd.read_csv(
    table_dir / "isnobal_watershed_daily.csv",
    parse_dates=["date"],
)

noah = pd.read_csv(
    table_dir / "noahmp_watershed_daily.csv",
    parse_dates=["date"],
)

for frame in (isnobal, noah):
    frame["water_year"] = (
        pd.to_numeric(frame["water_year"], errors="coerce")
        .astype("Int64")
    )

print("iSnobal rows:", len(isnobal))
print("Noah-MP rows:", len(noah))

isnobal.head()

## Example: replot one variable

In [ ]:
VARIABLE = "SWE"   # "SWE" or "Snow depth"

example_path, example_stats = plot_watershed_variable(
    cfg=cfg,
    isnobal=isnobal,
    noah=noah,
    variable=VARIABLE,
    show_figure=True,
)

# print("Saved figure:", example_path)
# example_stats

## Generate both figures

In [ ]:
all_stats = []
figure_paths = []

for variable in ["SWE", "Snow depth"]:
    path, stats = plot_watershed_variable(
        cfg=cfg,
        isnobal=isnobal,
        noah=noah,
        variable=variable,
        show_figure=False,
    )

    figure_paths.append(path)
    all_stats.append(stats)

stats_table = pd.concat(all_stats, ignore_index=True)

stats_path = table_dir / "watershed_yearly_stats_replot.csv"
stats_table.to_csv(stats_path, index=False)

print("Created figures:")
for path in figure_paths:
    print("  ", path)

print("\nCreated statistics table:")
print("  ", stats_path)

stats_table